In [1]:
# =========================================================================
# EXPERIMENT 3: SpectralFormer on Local Dataset + Indian Pines Few-Shot
# =========================================================================
# Paste this ENTIRE script into a NEW Kaggle notebook cell.
# Results for LightGBM, ViT, HybridSN already exist from Exp 2.
# This script ONLY runs SpectralFormer.
# =========================================================================
!pip install thop

# ==========================================
# IMPORTS & ENVIRONMENT
# ==========================================
import os
import gc
import copy
import time
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.io as sio
import urllib.request

from scipy.stats import ttest_rel, wilcoxon

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, cohen_kappa_score,
                             confusion_matrix, classification_report)

warnings.filterwarnings("ignore")

OUTPUT_DIR = "/kaggle/working/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        prop = torch.cuda.get_device_properties(i)
        print(f"  [{i}] {prop.name} - VRAM: {prop.total_memory / (1024**3):.2f} GB")


# ==========================================
# UTILITY FUNCTIONS
# ==========================================
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True


def calc_average_accuracy(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    class_accuracies = cm.diagonal() / cm.sum(axis=1)
    return np.nanmean(class_accuracies) * 100


def cohens_d(x, y):
    """Calculate Cohen's d effect size between two paired samples."""
    diff = np.array(x) - np.array(y)
    return np.mean(diff) / (np.std(diff, ddof=1) + 1e-10)


# ==========================================
# DATASET: SoilDataset (same as original)
# ==========================================
class SoilDataset(Dataset):
    def __init__(self, X, y, indices=None):
        self.X = X
        self.y = y
        self.indices = indices if indices is not None else np.arange(len(X))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        x = torch.as_tensor(self.X[real_idx], dtype=torch.float32)
        mean = x.mean()
        std = x.std() + 1e-8
        x = (x - mean) / std
        y_val = torch.as_tensor(self.y[real_idx], dtype=torch.long)
        return x, y_val


# ==========================================
# ARCHITECTURE: SpectralFormer
# ==========================================
# Reference: Hong et al., "SpectralFormer: Rethinking Hyperspectral Image
# Classification with Transformers," IEEE TGRS, 2022.
#
# Key Ideas:
# 1. Group-Wise Spectral Embedding (GSE): adjacent bands grouped into tokens.
# 2. Cross-Layer Adaptive Fusion (CAF): skip connections between layers.
# 3. CLS token for classification.
# ==========================================

class SpectralFormerBlock(nn.Module):
    """Single Transformer Encoder Block with Pre-LayerNorm."""
    def __init__(self, dim, heads=4, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=heads,
                                          dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        mlp_hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x_norm = self.norm1(x)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)
        x = x + attn_out
        x = x + self.mlp(self.norm2(x))
        return x


class CrossLayerAdaptiveFusion(nn.Module):
    """CAF module: learns to fuse features from current and previous layer."""
    def __init__(self, dim):
        super().__init__()
        self.alpha = nn.Parameter(torch.tensor(0.5))

    def forward(self, x_current, x_previous):
        alpha = torch.sigmoid(self.alpha)
        return alpha * x_current + (1.0 - alpha) * x_previous


class SpectralFormer(nn.Module):
    def __init__(self, bands=168, classes=3, group_size=5, dim=64,
                 depth=4, heads=4, dropout=0.1):
        super().__init__()
        self.bands = bands
        self.group_size = group_size
        self.num_tokens = bands // group_size
        self.effective_bands = self.num_tokens * group_size

        self.group_embed = nn.Linear(group_size, dim)
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_tokens, dim) * 0.02)
        self.cls_token = nn.Parameter(torch.randn(1, 1, dim) * 0.02)

        self.blocks = nn.ModuleList([
            SpectralFormerBlock(dim, heads=heads, mlp_ratio=4.0, dropout=dropout)
            for _ in range(depth)
        ])
        self.caf_modules = nn.ModuleList([
            CrossLayerAdaptiveFusion(dim)
            for _ in range(depth - 1)
        ])

        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, classes)

    def forward(self, x):
        if x.ndim == 4:
            b, h, w, c = x.shape
            x = x[:, h // 2, w // 2, :]
        B = x.shape[0]
        x = x[:, :self.effective_bands]
        x = x.view(B, self.num_tokens, self.group_size)
        x = self.group_embed(x)
        x = x + self.pos_embed
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1)

        prev_x = x
        for i, block in enumerate(self.blocks):
            x = block(x)
            if i > 0:
                x = self.caf_modules[i - 1](x, prev_x)
            prev_x = x

        x = self.norm(x[:, 0])
        return self.head(x)


# ==========================================
# TRAINING LOOP (with OOM prevention)
# ==========================================
def train_pytorch_model(model, train_loader, test_loader, epochs=100,
                        patience=10, lr=3e-4):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_loss = float('inf')
    best_model_state = None
    patience_counter = 0

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        correct_train = 0
        total_train = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            try:
                out = model(X_batch)
            except RuntimeError as e:
                if "out of memory" in str(e):
                    print(f"    OOM in training! Clearing cache.")
                    gc.collect()
                    torch.cuda.empty_cache()
                    continue
                raise e
            loss = criterion(out, y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            preds = torch.argmax(out, dim=1)
            correct_train += (preds == y_batch).sum().item()
            total_train += y_batch.size(0)

        if total_train == 0:
            continue
        train_loss = epoch_loss / max(len(train_loader), 1)
        train_acc = correct_train / total_train
        scheduler.step()

        # Validation
        model.eval()
        val_loss = 0
        correct_val = 0
        total_val = 0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                try:
                    out = model(X_batch)
                except RuntimeError as e:
                    if "out of memory" in str(e):
                        gc.collect()
                        torch.cuda.empty_cache()
                        continue
                    raise e
                val_loss += criterion(out, y_batch).item()
                preds = torch.argmax(out, dim=1)
                correct_val += (preds == y_batch).sum().item()
                total_val += y_batch.size(0)

        if total_val == 0:
            continue
        val_loss /= max(len(test_loader), 1)
        val_acc = correct_val / total_val

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        if val_loss < best_loss:
            best_loss = val_loss
            best_model_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"      Early stopping at epoch {epoch + 1}")
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    # Pure Inference
    model.eval()
    all_preds, all_y = [], []
    inf_start = time.time()
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            out = model(X_batch.to(device))
            preds = torch.argmax(out, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_y.extend(y_batch.numpy())
    inf_time = time.time() - inf_start

    acc = accuracy_score(all_y, all_preds)
    kappa = cohen_kappa_score(all_y, all_preds)
    return acc, kappa, np.array(all_preds), model, history, inf_time


# ==========================================
# TRAINING CURVES PLOTTING
# ==========================================
def plot_training_curves(history, model_name, split_label):
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss', color='blue')
    plt.plot(history['val_loss'], label='Val Loss', color='orange')
    plt.title(f"{model_name} Loss ({split_label})")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)

    plt.subplot(1, 2, 2)
    plt.plot(history['train_acc'], label='Train Acc', color='blue')
    plt.plot(history['val_acc'], label='Val Acc', color='orange')
    plt.title(f"{model_name} Accuracy ({split_label})")
    plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    fname = f"Figure_Training_Curves_{model_name}_{split_label}.png"
    plt.savefig(fname, dpi=300)
    plt.close('all')
    print(f"    Saved: {fname}")


# ==========================================
# DATA LOADING
# ==========================================
def read_bil_file(filepath, samples=320, bands=168, dtype=np.uint16):
    filesize = os.path.getsize(filepath)
    bytes_per_pixel = np.dtype(dtype).itemsize
    lines = filesize / (samples * bands * bytes_per_pixel)
    if not lines.is_integer():
        return None
    lines = int(lines)
    raw_data = np.fromfile(filepath, dtype=dtype)
    img_cube = raw_data.reshape((lines, bands, samples))
    img_cube = np.transpose(img_cube, (0, 2, 1))
    return img_cube


def extract_patches(img_cube, label, condition, patch_size=15):
    h, w, c = img_cube.shape
    patches, labels, conditions = [], [], []
    for i in range(0, h - patch_size, patch_size):
        for j in range(0, w - patch_size, patch_size):
            patch = img_cube[i:i + patch_size, j:j + patch_size, :]
            patches.append(patch)
            labels.append(label)
            conditions.append(condition)
    return patches, labels, conditions


def load_local_dataset(base_dir):
    print(f"Loading Local Dataset from: {base_dir}")
    X, y, cond = [], [], []
    bil_files = []
    for root, _, files in os.walk(base_dir):
        for f in files:
            if f.endswith('.bil'):
                bil_files.append(os.path.join(root, f))
    if len(bil_files) == 0:
        raise FileNotFoundError("No .bil files found!")
    print(f"  Found {len(bil_files)} .bil files.")
    for filepath in bil_files:
        root_dir = os.path.dirname(filepath)
        img = read_bil_file(filepath)
        if img is None:
            continue
        label = -1
        if 'black' in root_dir.lower(): label = 0
        elif 'red' in root_dir.lower(): label = 1
        elif 'yellow' in root_dir.lower(): label = 2
        condition = 1 if 'moist' in root_dir.lower() else 0
        if label != -1:
            p, l, c = extract_patches(img, label, condition)
            X.extend(p)
            y.extend(l)
            cond.extend(c)
    X = np.array(X, dtype=np.float32)
    y = np.array(y)
    cond = np.array(cond)
    print(f"  Local Data: {X.shape[0]} patches, shape {X.shape}")
    return X, y, cond


def load_indian_pines():
    print("Loading Indian Pines Dataset...")
    data_url = "https://raw.githubusercontent.com/gokriznastic/HybridSN/master/data/Indian_pines_corrected.mat"
    label_url = "https://raw.githubusercontent.com/gokriznastic/HybridSN/master/data/Indian_pines_gt.mat"
    data_path = "Indian_pines_corrected.mat"
    label_path = "Indian_pines_gt.mat"
    if not os.path.exists(data_path):
        urllib.request.urlretrieve(data_url, data_path)
    if not os.path.exists(label_path):
        urllib.request.urlretrieve(label_url, label_path)
    X_full = sio.loadmat(data_path)['indian_pines_corrected'].astype(np.float32)
    y_full = sio.loadmat(label_path)['indian_pines_gt']
    h, w, c = X_full.shape
    patch_size = 15
    margin = patch_size // 2
    img_padded = np.pad(X_full, ((margin, margin), (margin, margin), (0, 0)), mode='reflect')
    patches, labels = [], []
    for i in range(h):
        for j in range(w):
            if y_full[i, j] > 0:
                patches.append(img_padded[i:i + patch_size, j:j + patch_size, :])
                labels.append(y_full[i, j] - 1)
    X = np.array(patches, dtype=np.float32)
    y = np.array(labels)
    bands = X.shape[-1]
    classes = len(np.unique(y))
    print(f"  Indian Pines: {X.shape[0]} patches, {bands} bands, {classes} classes")
    return X, y, bands, classes


# ==================================================================
# EXPERIMENT A: SpectralFormer on Local Dataset (10-Seed)
# ==================================================================
def run_local_dataset_experiment(X_local, y_local, conditions,
                                 bands=168, classes=3, num_seeds=10,
                                 epochs=100, patience=20):
    print("\n" + "=" * 60)
    print(f"EXPERIMENT A: SpectralFormer on Local Dataset ({num_seeds}-Seed)")
    print("=" * 60)

    accs, kappas = [], []

    for seed in range(num_seeds):
        seed_everything(seed)
        print(f"\n  [Seed {seed + 1}/{num_seeds}]")

        # Same split as original: Train on Dry + 10% Moist, Test on 90% Moist
        moist_indices = np.where(conditions == 1)[0]
        dry_indices = np.where(conditions == 0)[0]
        train_moist, test_moist = train_test_split(
            moist_indices, test_size=0.8, random_state=seed
        )
        train_idx = np.concatenate([dry_indices, train_moist])
        test_idx = test_moist

        train_loader = DataLoader(SoilDataset(X_local, y_local, train_idx),
                                  batch_size=32, shuffle=True)
        test_loader = DataLoader(SoilDataset(X_local, y_local, test_idx),
                                 batch_size=32, shuffle=False)

        sf = SpectralFormer(bands=bands, classes=classes, group_size=4,
                            dim=64, depth=4, heads=4, dropout=0.1)
        acc, kappa, preds, _, history, inf_time = train_pytorch_model(
            sf, train_loader, test_loader, epochs=epochs, patience=patience
        )

        accs.append(acc * 100)
        kappas.append(kappa)
        print(f"    SpectralFormer -> Acc: {acc * 100:.2f}% | Kappa: {kappa:.4f}")

        if seed == 0:
            y_test = y_local[test_idx]
            print(f"\n  [SpectralFormer Classification Report (Seed 1)]")
            print(classification_report(y_test, preds))
            plot_training_curves(history, "SpectralFormer", "LocalDataset")

        del sf
        gc.collect()
        torch.cuda.empty_cache()

    # Final Results
    mean_acc = np.mean(accs)
    std_acc = np.std(accs)
    mean_kap = np.mean(kappas)
    std_kap = np.std(kappas)

    print("\n" + "=" * 60)
    print(f"FINAL: SpectralFormer on Local Dataset ({num_seeds}-Seed)")
    print(f"  OA:    {mean_acc:.2f} +/- {std_acc:.2f}%")
    print(f"  Kappa: {mean_kap:.4f} +/- {std_kap:.4f}")
    print("=" * 60)

    return {'acc': accs, 'kappa': kappas}


# ==================================================================
# EXPERIMENT B: SpectralFormer on Indian Pines Few-Shot (10-Seed)
# ==================================================================
def run_fewshot_experiment(X, y, bands=200, classes=16,
                           epochs=50, num_seeds=10):
    splits = [0.01, 0.02, 0.05, 0.10, 0.15, 0.20]

    curve_data_mean = []
    curve_data_std = []

    print("\n" + "=" * 70)
    print(f"EXPERIMENT B: {num_seeds}-SEED FEW-SHOT (SpectralFormer only)")
    print("=" * 70)

    for split in splits:
        print(f"\n{'=' * 50}")
        print(f"--- Training Split: {split * 100}% ---")
        print(f"{'=' * 50}")

        split_accs = []

        for seed in range(num_seeds):
            seed_everything(seed)
            print(f"  [Seed {seed + 1}/{num_seeds}]")

            train_idx, test_idx = train_test_split(
                np.arange(len(y)), train_size=split,
                random_state=seed, stratify=y
            )
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            train_loader = DataLoader(SoilDataset(X_train, y_train),
                                      batch_size=32, shuffle=True)
            test_loader = DataLoader(SoilDataset(X_test, y_test),
                                     batch_size=128, shuffle=False)

            sf = SpectralFormer(bands=bands, classes=classes, group_size=5,
                                dim=64, depth=4, heads=4, dropout=0.1).to(device)
            acc, _, preds, _, hist, _ = train_pytorch_model(
                sf, train_loader, test_loader, epochs=epochs, patience=15
            )
            split_accs.append(acc * 100)

            if seed == 0:
                print(f"\n  [SpectralFormer {int(split * 100)}% Classification Report]")
                print(classification_report(y_test, preds))
                plot_training_curves(hist, "SpectralFormer", f"{int(split * 100)}pct")

            del sf
            gc.collect()
            torch.cuda.empty_cache()

        mean_acc = np.mean(split_accs)
        std_acc = np.std(split_accs)
        curve_data_mean.append(mean_acc)
        curve_data_std.append(std_acc)
        print(f"  Final SpectralFormer -> OA: {mean_acc:.2f}% +/- {std_acc:.2f}%")

    # Plot
    plt.figure(figsize=(10, 6))
    splits_pct = [s * 100 for s in splits]
    plt.errorbar(splits_pct, curve_data_mean, yerr=curve_data_std,
                 fmt='-D', linewidth=2, label='SpectralFormer', color='#d62728', capsize=4)
    plt.title(f"Few-Shot ({num_seeds}-Seed): SpectralFormer on Indian Pines", fontsize=14, fontweight='bold')
    plt.xlabel("Training Data (%)", fontsize=12)
    plt.ylabel("Overall Accuracy (%)", fontsize=12)
    plt.xticks(splits_pct)
    plt.ylim(0, 100)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend(fontsize=11)
    plt.savefig("Figure_FewShot_SpectralFormer.png", dpi=300, bbox_inches='tight')
    plt.close('all')
    print("\nSaved: Figure_FewShot_SpectralFormer.png")


# ==================================================================
# FLOPs BENCHMARK
# ==================================================================
def benchmark_flops(bands=200, classes=16):
    from thop import profile, clever_format
    sf = SpectralFormer(bands=bands, classes=classes, group_size=5,
                        dim=64, depth=4, heads=4).to(device)
    dummy = torch.randn(1, 15, 15, bands).to(device)
    flops, params = profile(sf, inputs=(dummy,), verbose=False)
    flops_str, params_str = clever_format([flops, params], "%.2f")
    print(f"  SpectralFormer -> FLOPs: {flops_str} | Params: {params_str}")
    del sf
    gc.collect()
    torch.cuda.empty_cache()


# ==================================================================
# MAIN EXECUTION
# ==================================================================
if __name__ == '__main__':
    print("=" * 70)
    print("EXPERIMENT 3: SpectralFormer Benchmarking")
    print("=" * 70)

    LOCAL_SOIL_DIR = '/kaggle/input/datasets/dev123123456/local-soil-hyperspectral-dataset'

    # --- Load Data ---
    X_local, y_local, conditions = load_local_dataset(LOCAL_SOIL_DIR)
    X_indian, y_indian, indian_bands, indian_classes = load_indian_pines()

    # --- FLOPs ---
    print("\n--- FLOPs Benchmark ---")
    benchmark_flops(bands=indian_bands, classes=indian_classes)

    # --- Experiment A: Local Dataset (10-seed) ---
    run_local_dataset_experiment(
        X_local, y_local, conditions,
        bands=168, classes=3, num_seeds=10,
        epochs=100, patience=20
    )

    # --- Experiment B: Indian Pines Few-Shot (10-seed) ---
    run_fewshot_experiment(
        X_indian, y_indian,
        bands=indian_bands, classes=indian_classes,
        epochs=50, num_seeds=10
    )

    print("\n" + "=" * 70)
    print("ALL EXPERIMENTS COMPLETE!")
    print("=" * 70)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 90.3 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requ